Given a set of 26 currents, simulate and summarize the 26 twiss parameters

In [1]:
from current_twiss_map import *
import numpy as np
from beamline import lattice
from ebeam import beam
EXCEL_PATH = os.path.abspath(os.path.join(CURRENT_DIR, "../../beam_excel/Beamline_elements_3.xlsx"))
BEAM_ENERGY = 40.0  # MeV as the energy for a single electron
N_PARTICLES = 1000  # number of electrons in the beam for simulation
SEED = 42  # random seed for reproducibility
CURRENT_BOUNDS = (0.01, 1.5)  # current bound in the experiment heuristically

# ── Relativistic factors ─────────────────────────────────────────────
relat = lattice(1, fringeType=None)
relat.setE(E=BEAM_ENERGY)  # set the energy of the relativistic factor
GAMMA_REL = relat.gamma  # Lorentz factor as the mass ratio of the electron and the static electron
BETA_REL = relat.beta  # standardized electron velocity
NORM = GAMMA_REL * BETA_REL  # electron momentom ratio between the current electron and the static one

# ── Undulator matching targets (MkV FEL, K=1.2, λ_u=2.3 cm) ─────────
K = 1.2  # Undulator Parameter
LAMBDA_U = 2.3e-2  # wavelength, unit: m
## twiss parameters
BETA_YM = GAMMA_REL / (K * 2 * np.pi / LAMBDA_U)  # matched β_y at UND entrance, beam size in y direction
BETA_XM = 1.4  # m  — matched β_x at UND entrance, beam size at the x direction
ALPHA_XM = 0.47  # — Courant-Snyder α_x at UND entrance, beam size converge at the direction x
ALPHA_YM = 0.0  # — Courant-Snyder α_y at UND entrance, beam waist is achieved at the y direction

# ── Beam parameters ─────────────────────────────────────────────────
epsilon_n = 8.0  # π·mm·mrad  normalised emittance, inner attribute of electron gun
x_std = 0.8  # mm, initial standard deviation at x direction
y_std = 0.8  # mm, initial standard deviation at y direction
f_RF = 2856e6  # Hz, acceleration frequency
bunch_spread_ps = 2.0  # ps, beam length
energy_spread_pct = 0.5  # %, energy standard deviation inside the beam
h = 5e9  # 1/s  chirp, longitudinal chirp for beam compression

epsilon = epsilon_n / NORM  # geometric emittance
x_prime_std = epsilon / x_std  # alpha_x=0 at the initial point, so that sigma_x * sigma_x'=epsilon
y_prime_std = epsilon / y_std  # alpha_y=0 at the initial point, so that sigma_y * sigma_y'=epsilon
tof_std = bunch_spread_ps * 1e-9 * f_RF  # Time of Flight Standard Deviation, change from cycle to mili-cycle, measuring how much the electron bunch occupies the cycle
energy_std = energy_spread_pct * 10  # transform from percent to per mill

In [2]:
np.random.seed(31)
currents =np.random.uniform(low=0.01, high=1.5, size=26)    # 26 values

bl = ExcelElements(EXCEL_PATH).create_beamline()
ebeam_obj = ebeam_class()
PARTICLES = ebeam_obj.gen_6d_gaussian(
    0,
    [x_std, x_prime_std, y_std, y_prime_std, tof_std, energy_std],
    N_PARTICLES,
)
print(f"test currents: {currents}")

test currents: [0.43622019 1.43757729 1.15776627 1.4804363  0.32016654 0.2140064
 1.36347697 0.11227139 0.12224576 0.81986669 0.14320558 0.57976597
 1.00615506 0.64946184 0.0754949  0.29948612 0.67552263 0.10323418
 0.45337525 1.41601004 0.4313885  0.40886338 0.61675632 1.2407257
 0.76498399 0.41151832]


In [3]:
twiss9 = currents_to_twiss(bl, currents, PARTICLES)
print(twiss9)   # [alpha_x, alpha_y, alpha_z, beta_x, beta_y, beta_z, eps_x, eps_y, eps_z]

tensor([[-6.1611e+02,  1.0477e+02,  1.2450e-01],
        [-1.2017e+02,  6.9188e+02,  9.6375e-02],
        [-4.5835e+00,  7.3721e+01,  8.4902e+01]], dtype=torch.float64)


In [7]:
# reference current set
REF_CURRENTS = [0.8218, 1.0430, 3.8834, 2.2396, 4.9532, 3.4258, 4.6657,
                2.6942, 2.6523, 0.2768, 0.2768, 2.6523, 2.6942, 4.6739,
                3.1219, 3.3129, 5.1775, 4.0434, 4.6818, 3.9336, 4.0787,
                0.0139, 1.3624, 0.9452, 2.8851, 2.1921]
evaluation_pos=8
twiss9 = currents_to_twiss(bl, REF_CURRENTS, PARTICLES, evaluation_pos)
print(twiss9)

tensor([[5.6188e-02, 1.8572e+00, 1.0479e-01],
        [2.1405e+00, 1.1411e+00, 9.6375e-02],
        [1.1946e-02, 1.1779e+00, 2.9852e+01]], dtype=torch.float64)


In [8]:
for i in range(1000):
    currents=np.random.uniform(low=0.01, high=1.5, size=26)
    print(f"test {i+1}: {currents_to_twiss(bl, currents, PARTICLES)}")

test 1: tensor([[-2.4959e+12,  4.1519e+11,  3.1623e-08],
        [-1.2906e+05,  5.8119e+05,  9.6659e-02],
        [-4.2294e-01,  1.1234e+02,  2.4153e+03]], dtype=torch.float64)
test 2: tensor([[ 9.2153e-01,  1.2866e-02,  1.0479e-01],
        [-1.4425e+04,  6.4454e+04,  9.6381e-02],
        [-5.2078e-01,  1.6541e+00,  3.2982e+01]], dtype=torch.float64)
test 3: tensor([[-3.3691e+04,  5.8291e+03,  1.0590e-01],
        [-8.5830e+04,  3.0915e+05,  9.6414e-02],
        [-4.9825e-02,  2.0699e+01,  5.2336e+02]], dtype=torch.float64)
test 4: tensor([[-2.4577e+02,  3.8595e+01,  1.0457e-01],
        [-4.1715e+04,  2.3654e+05,  9.6391e-02],
        [ 2.3318e-01,  2.3235e+00,  5.5857e+01]], dtype=torch.float64)
test 5: tensor([[-1.4413e+13,  2.5057e+12,  3.1623e-08],
        [-1.0624e+04,  5.6514e+04,  9.6375e-02],
        [ 9.7823e-02,  1.9157e+02,  4.8097e+03]], dtype=torch.float64)
test 6: tensor([[-3.6819e+01,  1.2400e-01,  1.0535e-01],
        [-3.4996e+05,  1.7585e+06,  9.8016e-02],
        [